# Lesson 5c — Automation: Many Files — Solutions

> **Teacher reference** — This notebook contains worked solutions. Do not share with students.


One of three independent Lesson 5 modules — **5a fitting · 5b statistics · 5c automation** — do them in any order.

This notebook shows how to **automate a repetitive analysis** over many files using a loop, a reusable function, and `glob` — so you write the analysis once and run it on every file in a folder.

## Automation: many files

So far we worked with one table at a time. In real research you often have **many** files — one per test:

```text
runs/S01.csv
runs/S02.csv
runs/S03.csv
...
```

Doing the same analysis by hand on each file is slow and error-prone. This is where Python clearly beats a manual spreadsheet workflow: write the analysis **once**, run it on **every** file.

In [ ]:
# Part C — prepare the example files, then discover them
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import glob

def load_table(name):
    """Robust CSV loader: works locally and in the browser (JupyterLite).
    Do not worry about the details — this is just plumbing."""
    for folder in ("data", "../data", "."):
        try:
            return pd.read_csv(f"{folder}/{name}")
        except Exception:
            pass
    import js
    from pyodide.http import open_url
    base = str(js.location.href).split("/extensions/")[0]
    return pd.read_csv(open_url(f"{base}/files/data/{name}"))

# In real research these per-specimen files would already exist (one per test).
# Here we create them from the master table so the example is self-contained.
RUNS_DIR = Path("runs")
RUNS_DIR.mkdir(exist_ok=True)
master = load_table("tensile_raw.csv")
for sid, group in master.groupby("sample_id"):
    group.to_csv(RUNS_DIR / f"{sid}.csv", index=False)

# glob finds all files matching a pattern in a folder
files = sorted(glob.glob(str(RUNS_DIR / "*.csv")))
print(f"Found {len(files)} files:")
for f in files:
    print("  ", f)

In [ ]:
# Define a function that analyses ONE tensile-test file
def analyse_tensile_file(file_path):
    df = pd.read_csv(file_path)
    df["area_mm2"] = np.pi * (df["diameter_mm"] / 2) ** 2
    df["stress_MPa"] = df["force_N"] / df["area_mm2"]
    df["strain"] = df["displacement_mm"] / df["gauge_length_mm"]

    peak = df["stress_MPa"].idxmax()
    return {
        "sample_id": df["sample_id"].iloc[0],
        "treatment": df["treatment"].iloc[0],
        "UTS_MPa": df.loc[peak, "stress_MPa"],
        "strain_at_UTS": df.loc[peak, "strain"],
        "n_points": len(df),
    }

In [ ]:
# Loop over all files and collect one row per specimen
records = []
for f in files:
    records.append(analyse_tensile_file(f))

all_results = pd.DataFrame(records)
all_results

In [ ]:
# Save the automated summary table
output_path = Path("automated_tensile_summary.csv")
all_results.to_csv(output_path, index=False)
print("Saved:", output_path.resolve())

In [ ]:
# Plot every specimen's curve with a single loop
plt.figure(figsize=(7, 4))
for f in files:
    df = pd.read_csv(f)
    df["stress_MPa"] = df["force_N"] / (np.pi * (df["diameter_mm"] / 2) ** 2)
    df["strain"] = df["displacement_mm"] / df["gauge_length_mm"]
    plt.plot(df["strain"], df["stress_MPa"], label=df["sample_id"].iloc[0])

plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title("All tensile-test files, plotted automatically")
plt.legend(ncol=2)
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 1 — Add a column to the pipeline

Extend the analysis so the summary table carries one more quantity.

1. copy `analyse_tensile_file()` into the cell below and add **one new key** to the returned dict —
   e.g. the minimum stress, the final strain, or the mean stress;
2. re-run the loop over `files` to rebuild `all_results`;
3. save the updated table with `to_csv`.

**Check:** does the new column appear when you display `all_results`?

In [ ]:
# Exercise 1 — SOLUTION

def analyse_tensile_file(file_path):
    df = pd.read_csv(file_path)
    df["area_mm2"]   = np.pi * (df["diameter_mm"] / 2) ** 2
    df["stress_MPa"] = df["force_N"] / df["area_mm2"]
    df["strain"]     = df["displacement_mm"] / df["gauge_length_mm"]

    peak = df["stress_MPa"].idxmax()
    return {
        "sample_id":      df["sample_id"].iloc[0],
        "treatment":      df["treatment"].iloc[0],
        "UTS_MPa":        df.loc[peak, "stress_MPa"],
        "strain_at_UTS":  df.loc[peak, "strain"],
        "n_points":       len(df),
        "min_stress_MPa": df["stress_MPa"].min(),   # new column
    }

records    = [analyse_tensile_file(f) for f in files]
all_results = pd.DataFrame(records)
all_results.to_csv("automated_tensile_summary.csv", index=False)
all_results


### Exercise 2 — Query the results table

Once the analysis is automated, `all_results` is just a normal pandas table — you can explore it with the same tools from Lesson 3.

1. sort `all_results` by `UTS_MPa` from highest to lowest and print it;
2. which specimen is the strongest?

**Bonus:** print only the rows where `UTS_MPa` is above 400 MPa.

In [ ]:
# Exercise 2 — SOLUTION

sorted_results = all_results.sort_values("UTS_MPa", ascending=False)
print(sorted_results.to_string(index=False))
# S03 (quenched) is the strongest specimen — highest UTS.

# Bonus: specimens with UTS > 400 MPa
print()
print("Specimens with UTS > 400 MPa:")
print(all_results[all_results["UTS_MPa"] > 400].to_string(index=False))

## Closing — Course recap

This final lesson ties together the whole course:

```text
L1  ->  Foundations & first look
        variables, types, lists, conditions, loops, functions; a read->compute->plot demo

L2  ->  Python basics (hands-on)
        arithmetic, booleans, if/elif, lists, for, functions, dictionaries, basic plots

L3  ->  From raw files to clean data tables
        read_csv, inspect, filter, calculated columns, cleaning, groupby, save

L4  ->  Plotting and data analysis
        line/scatter/hist/box/bar plots, summary metrics, a linear fit

L5  ->  Three practical tools
        nonlinear fitting, uncertainty & statistics, automation over many files
```

The central message:

```text
Python pays off when the analysis must be repeated, checked, modified, or shared.
```

## Where to go next

Three useful directions after this introductory course:

## 1. SciPy
Fitting, optimization, statistics, interpolation, integration, signal processing.
Start with: `scipy.optimize.curve_fit`, `scipy.stats`, interpolation.

## 2. pandas
Tabular data analysis.
Start with: missing values, merging tables, grouping/aggregation, reshaping.

## 3. Matplotlib
Publication-quality figures.
Start with: `fig, ax = plt.subplots()`, axis customization, saving vector graphics, multi-panel figures.

Final practical advice:

```text
Start with small notebooks that solve real problems from your own research.
```